### Setup

These cells are copied unchanged from Section 1 of `exam2.ipynb`: the imports, the Black–Scholes functions and sample data of §1.4, and the simulation code of §1.6–§1.8. Run them before the Results below.

In [ ]:
from hedging_harness.notebook_views import run
_ = run('MonteCarlo_StochasticVolatility')


#### 2.5.1 Stochastic volatility: how the schemes cope when $\sigma$ is random

So far $\sigma$ has been a fixed number. Lecture 7 (p.6) leaves open the question "What if it's random?". Here we let
it be random and repeat the scheme comparison of §2.2. Under geometric Brownian motion (GBM) the exact scheme of §1.8
has no discretisation error. With random volatility none of our schemes is exact, and this is the case for which §1.5
reserves Euler–Maruyama and Milstein.

We use the Heston model. The variance $v$ (volatility squared) gets its own equation. It is pulled back towards
a long-run level $\theta$ at speed $\kappa$, so it cannot drift off for ever. It is shaken by its own Brownian motion,
scaled by $\xi$, the volatility of variance. Our first comparison sets the correlation of the two Brownian motions to $\rho=-0.7$, so
volatility tends to rise when the stock falls. We then vary this correlation separately. Under $\mathbb{Q}$:

$$
dS = rS\,dt + \sqrt{v}\,S\,dX_1, \qquad
dv = \kappa(\theta - v)\,dt + \xi\sqrt{v}\,dX_2, \qquad
\mathrm{Corr}(dX_1, dX_2) = \rho .
$$

We keep $S_0$, $K$, $T$ and $r$ at the sample values and set $v_0=\theta=0.04$.
Expected variance then remains 0.04, matching the variance of the constant 20% volatility benchmark;
expected volatility itself need not equal 20%. With $\xi=0$, the variance stays constant and we recover GBM.

**The first input varied here is $\xi$, the volatility of variance.** The two cases use $\xi=0.3$ and $0.7$.
As introduced in §1.11, the Feller condition compares the pull away from zero with the size of variance
shocks. At the chosen $\kappa$ and $\theta$, $2\kappa\theta=0.16$: the condition holds when $\xi=0.3$
($\xi^2=0.09$) and fails when $\xi=0.7$ ($\xi^2=0.49$).

| Table/figure label | Variance fluctuations | $\kappa$ | $\theta=v_0$ | $\xi$ | $\rho$ | $2\kappa\theta/\xi^2$ |
|---|---|---|---|---|---|---|
| Feller satisfied | Lower | 2 | 0.04 | 0.3 | −0.7 | 1.78 |
| Feller violated | Higher | 2 | 0.04 | 0.7 | −0.7 | 0.33 |

In the second case, zero is attainable in the continuous variance process, making the numerical treatment
near zero especially relevant. The continuous process remains non-negative in both cases. Discrete
approximations can produce negative variance even in the first case; satisfying the Feller condition
does not guarantee that a numerical scheme preserves positivity.

**What we measure, and against what.**

- **Benchmark (Table 2.5.1a).** Heston's semi-closed form, $C = S_0P_1 - Ke^{-r(T-t)}P_2$, is the counterpart of the
  Black–Scholes formula of §1.4: $P_2 = \mathbb{Q}(S_T > K)$ plays the role of $N(d_2)$. $P_1$ and $P_2$ come from
  integrating the characteristic function of $\ln S_T$ numerically. Before we rely on it, we check it against
  Black–Scholes as $\xi \to 0$, against a second, independent pricing formula, and against a fine-step Monte Carlo price.
- **Weak error**, the bias of the call price, against $\delta t$ for each scheme, under GBM and both Heston sets:
  Table 2.5.1b and Figure 2.5.1a. The table retains monthly and daily values; the figure shows the full ladder.
- **Strong error relative to a reference**, how far each terminal price lands from the exact GBM
  solution or the fine-grid Heston approximation driven by the same Brownian path, and the share of steps on
  which the variance goes negative: Table 2.5.1c and Figure 2.5.1b.
- **The smile.** We compare implied volatility across strikes with the flat 20% Black–Scholes
  benchmark. Implied volatility is the Black–Scholes volatility that reproduces a given Heston price
  (Lecture 7, pp.61–63): Figure 2.5.1c. The left panel keeps the original $\xi$ comparison. The right
  panel holds $\xi=0.3$ and all other inputs fixed, changing only $\rho$ to $-0.7$, $0$ and $+0.7$.
  The Feller condition therefore remains satisfied. This asks how much the downward slope depends on
  our negative-correlation choice. Positive correlation is included as a comparison, not a market calibration.
  Selected Monte Carlo points use the log scheme, common random numbers and 100,000 paths on 504 steps;
  paired 252-step paths provide a check on time-step sensitivity. The intervals show sampling uncertainty only.

**What §1.8 predicts.** Under GBM: Euler strong order ½ and weak order 1, Milstein strong and weak order 1, and no
error for the exact scheme. For the Heston approximation used here, omitting part of the two-noise interaction can remove
Milstein's order-1 advantage. The stock's volatility now
depends on a second Brownian motion, and a full Milstein step needs the iterated integral
$\int\!\!\int dX_2\,dX_1$, which the increments $\delta X_1, \delta X_2$ do not determine (Kloeden and Platen,
1992). The square-root coefficient is not differentiable at zero; when that boundary is attainable,
the regularity assumptions behind the usual convergence orders need particular care.


**The schemes under Heston.** Each scheme of §1.8 is applied to the pair $(S, v)$. The two Brownian increments are
built from independent standard normal draws $\phi, \phi'$ so that they have correlation $\rho$. Only the positive part
of the variance, $v_n^+ = \max(v_n, 0)$, is ever used inside the coefficients.

- **Euler** steps $S$ as in §1.8 and $v$ with the *full truncation* fix of Lord, Koekkoek and van Dijk (2010): $v$ may
  go negative, but only its positive part drives the next step. They found it the most accurate of the biased
  schemes they compared.
- **Milstein** adds the $\tfrac12 b\,b'$ correction of §1.8 to each equation, and a cross term to the $S$ step because
  $v$ moves inside $\sqrt{v}\,S$. The Lévy-area part of that cross term is dropped. It describes the interaction of the
  two Brownian motions within the step, which their endpoint increments alone do not determine. $v$ is floored at zero after
  each step.
- **Milstein-FT** (`milstein_ft`) is a diagnostic variant: Milstein's step for $S$ with Euler's step for $v$. This changes both the variance update and its treatment near zero, so the comparison
  does not isolate the effect of flooring alone.
- **log** is the exact step of §1.8 for $S$ with the variance frozen over the step, and Euler's step for $v$. With
  $\xi = 0$ it *is* the exact scheme of §1.8. Exact simulation of Heston (Broadie and Kaya, 2006) is far more
  involved and is beyond the present study.

In symbols, the increments, the three steps for $S$ and the two steps for $v$ are

$$
\begin{aligned}
&\delta X_1 = \sqrt{\delta t}\,\phi, \qquad \delta X_2 = \rho\,\delta X_1 + \sqrt{1-\rho^2}\,\sqrt{\delta t}\,\phi', \\
\text{Euler:}\quad &\hat S_{n+1} = \hat S_n + r\hat S_n\,\delta t + \sqrt{v_n^+}\,\hat S_n\,\delta X_1, \\
\text{Milstein:}\quad &\hat S_{n+1} = \hat S_n + r\hat S_n\,\delta t + \sqrt{v_n^+}\,\hat S_n\,\delta X_1
  + \tfrac12 v_n^+\hat S_n\left(\delta X_1^2 - \delta t\right) + \tfrac14\xi\hat S_n\left(\delta X_1\delta X_2 - \rho\,\delta t\right), \\
\text{log:}\quad &\ln \hat S_{n+1} = \ln \hat S_n + \left(r - \tfrac12 v_n^+\right)\delta t + \sqrt{v_n^+}\,\delta X_1, \\
\text{full truncation } v:\quad &v_{n+1} = v_n + \kappa(\theta - v_n^+)\,\delta t + \xi\sqrt{v_n^+}\,\delta X_2, \\
\text{Milstein } v:\quad &v_{n+1} = \max\!\left(v_n^+ + \kappa(\theta - v_n^+)\,\delta t + \xi\sqrt{v_n^+}\,\delta X_2
  + \tfrac14\xi^2\left(\delta X_2^2 - \delta t\right),\; 0\right).
\end{aligned}
$$

**Measuring a bias smaller than the noise: common random numbers.** Many of the biases in Table 2.5.1b are smaller
than the standard error of a price from $N = 200{,}000$ paths, which is 0.02 to 0.03. Comparing two independent
prices would not show them. So every path is first simulated on a fine grid of $n_{\text{ref}} = 3{,}276$ steps (13
per trading day). The same path on a coarse grid of $n$ steps is built by adding up the fine increments in blocks of
$n_{\text{ref}}/n$. The weak error at $n$ steps is then the average, over the same paths, of the coarse discounted
payoff minus the fine one. The two payoffs move together, so their difference is far less noisy than either
price. Under GBM the reference is the exact solution on the same path. Under Heston it is the log scheme on the
fine grid, checked in Table 2.5.1a; a first-order extrapolation of its bias is printed under Table 2.5.1b. This estimate is
smaller than the daily standard errors, but is not an independent bound on the reference error. All models, schemes and step counts ($n$ = 3, 6, 12, 26, 52, 126, 252: four-monthly to daily) share the same
Brownian paths, seeded from `SEED + 251`.


**Reading the results.** The independent pricing checks agree closely (Table 2.5.1a).
With $\xi=0.3$ (“Feller satisfied”), the implemented Milstein approximation retains a smaller terminal-path
error than Euler, but much less of the advantage observed under GBM. Its fitted order is 0.57 relative to
the fine-grid reference (Table 2.5.1c).

With $\xi=0.7$ (“Feller violated”), the variance treatment becomes more consequential. At daily steps,
the floored Milstein call-price difference from the reference is +0.0630; the Milstein-FT variant gives
+0.0096 and is not resolved from zero at the 95% level (Table 2.5.1b). These implementations also differ in
their variance update, so the comparison does not isolate flooring alone. Euler's numerical variance is
negative on 10.01% of daily steps in this case.

Figure 2.5.1c shows the change in the pricing distribution. In the left panel, implied volatility is no longer
flat across strikes; for the higher $\xi$ it falls from about 27.8% at $K=70$ to 13.2% at $K=130$. Matching
the mean variance of the Black–Scholes benchmark does not reproduce its option prices.

The right panel changes only correlation. The downward slope at $\rho=-0.7$ becomes much flatter at zero
correlation and slopes upwards at $\rho=+0.7$. This helped me separate the effect of random variance from
the effect of linking variance and stock shocks: the original downward slope depended on both choices.
The largest of the nine Monte Carlo price discrepancies is 2.11 standard errors from the benchmark,
slightly outside its pointwise 95% interval. The paired change from 252 to 504 steps, allowing for its
own sampling uncertainty, is below 0.13 of a pricing standard error at every checked point. This supports
using the curves to interpret correlation here, without treating the simulation as exact.
